# Ask Coresight: search model v3 (about 40-50 minutes on an A100)

v2 found the right **topic** but often the wrong **kind** of question ("which dollar stores" → "have you bought at a dollar
store"), and its confidence did not tell right from wrong. v3 adds the three techniques the large labs use:
1. **Practice questions** written by a free open model (Qwen2.5-7B-Instruct) that keep the kind of answer (list, yes/no, reason, amount).
2. **Look-alikes** from the same topic kept as wrong answers only when a teacher scores them clearly lower.
3. **A teacher** (bge-reranker-v2-m3) whose scores the small model learns to copy.

1. Runtime → Change runtime type → **A100** GPU → Save. 2. Runtime → **Run all**; at **Choose Files** pick `catalogue.json`
(and `synthetic.json` too if an earlier run downloaded it: the practice questions are then not written again).
3. **Keep the laptop open.** `synthetic.json` downloads after step 4 (keep it), `bge-cip-v3.zip` at the end; the GPU is released 15 minutes later.

In [ ]:
# quiet: Colab's own packages (gradio, diffusers) are not used here, so pip's notes about them are hidden
!pip -q install "sentence-transformers==3.3.1" accelerate onnx onnxscript onnxruntime > /dev/null 2>&1   # 3.3.1: trained here on 12 Nov
import torch, json, random, time, os, re, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none: Runtime → Change runtime type → GPU")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from google.colab import files
if not os.path.exists("catalogue.json"):
    files.upload()   # choose catalogue.json, and synthetic.json if you have it from an earlier run
data = json.load(open("catalogue.json"))
docs, golden = data["docs"], data["golden"]
TRAIN_ASKS = 6   # asks 1-6 train and index; asks 7-8 and the golden questions are the test, never trained on
print(len(docs), "survey questions,", len(golden), "golden questions,", "synthetic.json found" if os.path.exists("synthetic.json") else "no synthetic.json yet")

In [ ]:
from sentence_transformers import SentenceTransformer

def index_texts():
    owners, texts = [], []
    for n, d in enumerate(docs):
        for t in [d["embed"]] + d["asks"][:TRAIN_ASKS]:
            owners.append(n); texts.append(t)
    return np.array(owners), texts

def ranked(encode, queries, k=20):
    """For each query: its k nearest survey questions (by their best text) and their closeness."""
    owners, texts = index_texts()
    vec = encode(texts); out = []
    for s in range(0, len(queries), 2048):   # in chunks: 80,000 questions × 25,000 texts at once would need 8 GB
        for row in encode(queries[s:s + 2048]) @ vec.T:
            per = np.full(len(docs), -1.0); np.maximum.at(per, owners, row)
            top = np.argsort(-per)[:k]; out.append((top, per[top]))
    return out

def tests():
    return {"unseen": ([a for d in docs for a in d["asks"][TRAIN_ASKS:]], [n for n, d in enumerate(docs) for _ in d["asks"][TRAIN_ASKS:]]),
            "golden": ([g["q"] for g in golden], golden)}

def right(name, top, want):
    return docs[top]["group"] == docs[want]["group"] if name == "unseen" else any(w in docs[top]["text"].lower() for w in want["want"])

def score(encode, label):
    """Share right on unseen asks and golden, and at each confidence gap (best vs next *different* question)."""
    out = {}
    for name, (queries, wants) in tests().items():
        rows = []
        for (top, sc), want in zip(ranked(encode, queries), wants):
            other = next((i for i in range(1, len(top)) if docs[top[i]]["group"] != docs[top[0]]["group"]), 1)
            rows.append((sc[0] - sc[other], right(name, top[0], want)))
        rows = np.array(rows, float)
        curve = {g: (float(rows[rows[:, 0] >= g, 1].mean()) if (rows[:, 0] >= g).any() else 0.0, float((rows[:, 0] >= g).mean()))
                 for g in (0.0, 0.02, 0.04, 0.06, 0.08, 0.10)}
        out[name] = curve
        print(f"{label:30} {name:7} right {curve[0.0][0]:.1%} · " + " · ".join(f"gap≥{g}: {r:.1%} on {c:.0%}" for g, (r, c) in list(curve.items())[1:]))
    return out

def st_encoder(model):
    return lambda texts: model.encode(texts, batch_size=256, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

base = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
results = {"as shipped": score(st_encoder(base), "bge-small as shipped")}

In [ ]:
# 1. Practice questions that keep what kind of answer the survey question gives, written by Qwen2.5-7B-Instruct (Apache 2.0)
if not os.path.exists("synthetic.json"):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = "Qwen/Qwen2.5-7B-Instruct"
    tok = AutoTokenizer.from_pretrained(name, padding_side="left")
    llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16, device_map="cuda")
    RULES = ("You write what people type into a consumer-survey search box. For the survey question below, write 12 different "
             "short questions someone would type to get THIS question's figures. Keep what kind of answer it gives: if it lists "
             "options (which retailers, which products), ask which/what/top; if it is yes or no, ask whether/how many people/what share; "
             "if it asks why, ask about reasons; if it asks an amount, ask how much. Never ask something another kind of question would "
             "answer. Mix: an analyst (precise, may add a group like Gen Z or income, or over time), a retail client (a decision), "
             "a colleague (casual, may have a typo), about one of its answer options. Plain words, never copy the survey wording, no numbers "
             "from the answers. Reply with a JSON list of 12 strings only.")
    def prompt(d):
        options = "; ".join(l.split(": ")[-1][:40] for l in d["embed"].split("| answers: ")[-1].split("; ")[:8]) if "| answers:" in d["embed"] else ""
        msg = f"Survey question: {d['text'][:300]}\nAnswer options: {options}"
        return tok.apply_chat_template([{"role": "system", "content": RULES}, {"role": "user", "content": msg}], tokenize=False, add_generation_prompt=True)
    synthetic, started = {}, time.time()
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["text"]))
    for s in range(0, len(order), 64):
        batch = order[s:s + 64]
        enc = tok([prompt(docs[i]) for i in batch], return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=380, do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tok.eos_token_id)
        for i, seq in zip(batch, out[:, enc["input_ids"].shape[1]:]):
            text = tok.decode(seq, skip_special_tokens=True)
            try:
                asks = [a.strip() for a in json.loads(text[text.index("["):text.rindex("]") + 1]) if isinstance(a, str) and 8 < len(a.strip()) < 160]
            except ValueError:
                asks = [m.strip() for m in re.findall(r'"([^"]{8,160})"', text)]
            synthetic[docs[i]["key"]] = asks[:12]
        print(f"{min(s + 64, len(order))} of {len(order)} · {time.time() - started:.0f}s", end=" | ")
    del llm; torch.cuda.empty_cache()
    json.dump(synthetic, open("synthetic.json", "w"))
    files.download("synthetic.json")   # keep it: a later run uploads it and skips this step
synthetic = json.load(open("synthetic.json"))
for d in docs:
    d["extra"] = synthetic.get(d["key"], [])
print("\npractice questions:", sum(len(d["extra"]) for d in docs), "for", sum(bool(d["extra"]) for d in docs), "survey questions")
for d in random.Random(3).sample(docs, 3):
    print("·", d["text"][:90], "→", d["extra"][:3])

In [ ]:
# 2. The teacher: reads a typed question and a survey question together. First, is it better than today's model here?
from sentence_transformers import CrossEncoder
teacher = CrossEncoder("BAAI/bge-reranker-v2-m3", device=DEVICE, max_length=256, default_activation_function=torch.nn.Identity(),
                       automodel_args={"torch_dtype": torch.float16})   # half precision: ~2x faster on the GPU, the same order
def teacher_scores(pairs):
    return teacher.predict(pairs, batch_size=512, show_progress_bar=False, convert_to_numpy=True)
check = {}
for name, (queries, wants) in tests().items():
    sample = list(zip(queries, wants))[:2000] if name == "unseen" else list(zip(queries, wants))
    tops = ranked(st_encoder(base), [q for q, _ in sample], k=20)
    base_right = teacher_right = 0
    for (q, want), (top, _) in zip(sample, tops):
        s = teacher_scores([(q, docs[t]["embed"]) for t in top])
        base_right += right(name, top[0], want); teacher_right += right(name, top[int(np.argmax(s))], want)
    check[name] = (base_right / len(sample), teacher_right / len(sample))
    print(f"{name}: today's model picks right {check[name][0]:.1%} · teacher reordering its top 20 {check[name][1]:.1%}")
results["teacher check"] = check
TEACHER_HELPS = check["golden"][1] > check["golden"][0] and check["unseen"][1] > check["unseen"][0]
print("teacher is better: its scores will be learned" if TEACHER_HELPS else "TEACHER IS NOT BETTER HERE: v3b is skipped, only v3a is trained")

In [ ]:
# 3. Look-alikes, NV-Retriever style: from the 10 nearest *different* questions, kept as a wrong answer only when the
# teacher scores it clearly below the right one (95% of the right one's probability); the teacher's margin is kept to be learned
anchors = [(t, n) for n, d in enumerate(docs) for t in d["asks"][:TRAIN_ASKS] + [d["name"]] + d["aliases"] + d["extra"] if t
           for _ in range(1 + (d["waves"] >= 10) + (d["waves"] >= 50))]   # weekly tracker questions count up to 3x
tops = ranked(st_encoder(base), [a for a, _ in anchors], k=12)
negs, margins, started = [], [], time.time()
for s in range(0, len(anchors), 2048):
    chunk = list(zip(anchors[s:s + 2048], tops[s:s + 2048]))
    pairs, spans = [], []
    for (t, n), (top, _) in chunk:
        cands = [int(c) for c in top if docs[c]["group"] != docs[n]["group"]][:10]
        spans.append((len(pairs), cands)); pairs += [(t, docs[n]["embed"])] + [(t, docs[c]["embed"]) for c in cands]
    logits = teacher_scores(pairs)
    for (at, cands), ((t, n), _) in zip(spans, chunk):
        pos = logits[at]; p_pos = 1 / (1 + np.exp(-pos))
        kept = [(c, l) for c, l in zip(cands, logits[at + 1:at + 1 + len(cands)]) if 1 / (1 + np.exp(-l)) < 0.95 * p_pos]
        c, l = max(kept, key=lambda x: x[1]) if kept else ((n + 1) % len(docs), pos - 10.0)
        negs.append(c); margins.append(float(pos - l))
    print(f"{min(s + 2048, len(anchors))} of {len(anchors)} · {time.time() - started:.0f}s", end=" | ")
print("\n", len(anchors), "training pairs, each with a teacher-checked look-alike")

In [ ]:
def train(epochs, distill, lr=3e-5, batch=128, scale=20.0, weight=0.5):
    """In-batch contrastive training with one teacher-checked look-alike per typed question (one question group a batch,
    never a near-twin as a wrong answer); with distill, also the teacher's margin between the right question and the
    look-alike (MarginMSE), so the small model's confidence follows the teacher's."""
    model = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEVICE)
    items = list(range(len(anchors)))
    steps = []
    for e in range(epochs):
        random.Random(e).shuffle(items)
        current, groups = [], set()
        for i in items:
            n, h = anchors[i][1], negs[i]
            if docs[n]["group"] in groups or docs[h]["group"] in groups: continue
            current.append(i); groups.add(docs[n]["group"])
            if len(current) == batch:
                steps.append(current); current, groups = [], set()
        if len(current) > 1:
            steps.append(current)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    warm = max(1, len(steps) // 10)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * max(0.0, 1 - s / len(steps)))
    model.train()
    emb = lambda ts: torch.nn.functional.normalize(model({k: v.to(DEVICE) for k, v in model.tokenize(ts).items() if torch.is_tensor(v)})["sentence_embedding"], dim=-1)
    started = time.time()
    for step in steps:
        a = emb([anchors[i][0] for i in step])
        p = emb([docs[anchors[i][1]]["embed"] for i in step]); h = emb([docs[negs[i]]["embed"] for i in step])
        loss = torch.nn.functional.cross_entropy(a @ torch.cat([p, h]).T * scale, torch.arange(len(step), device=DEVICE))
        if distill:
            student = ((a * p).sum(-1) - (a * h).sum(-1)) * scale
            loss = loss + weight * torch.nn.functional.mse_loss(student, torch.tensor([margins[i] for i in step], device=DEVICE, dtype=student.dtype))
        loss.backward(); opt.step(); sched.step(); opt.zero_grad()
    print(f"trained {epochs} epoch(s){' + teacher' if distill else ''} in {time.time() - started:.0f}s")
    model.eval()
    return model

models = {}
for label, distill in [("v3a", False), ("v3b teacher", True)]:
    if distill and not TEACHER_HELPS: continue
    models[label] = train(3, distill)
    results[label] = score(st_encoder(models[label]), label)

In [ ]:
# Every candidate shrunk to the server's format (8-bit ONNX, first-token vector) and checked with the server's code path;
# the choice is made in Ask's full pipeline afterwards
import onnxruntime, tokenizers, shutil
from onnxruntime.quantization import quantize_dynamic, QuantType
os.makedirs("bge-cip-v3", exist_ok=True)
for label, model in models.items():
    folder = "bge-cip-v3/" + label.replace(" ", "-")
    os.makedirs(folder, exist_ok=True)
    bert = model[0].auto_model.cpu().eval()
    dummy = model.tokenize(["a question"])
    torch.onnx.export(bert, (dummy["input_ids"], dummy["attention_mask"], dummy["token_type_ids"]), "full.onnx",
                      input_names=["input_ids", "attention_mask", "token_type_ids"], output_names=["last_hidden_state"],
                      dynamic_axes={k: {0: "batch", 1: "tokens"} for k in ["input_ids", "attention_mask", "token_type_ids", "last_hidden_state"]},
                      opset_version=17, dynamo=False)
    quantize_dynamic("full.onnx", folder + "/model_optimized.onnx", weight_type=QuantType.QInt8)
    model.tokenizer.save_pretrained("tok"); shutil.copy("tok/tokenizer.json", folder + "/tokenizer.json")
    words = tokenizers.Tokenizer.from_file(folder + "/tokenizer.json")
    words.enable_truncation(512); words.enable_padding(pad_id=0, pad_token="[PAD]")
    options = onnxruntime.SessionOptions(); options.intra_op_num_threads = options.inter_op_num_threads = 1
    session = onnxruntime.InferenceSession(folder + "/model_optimized.onnx", options, providers=["CPUExecutionProvider"])
    def onnx_encode(texts):   # api/ask_model.py embed, line for line
        out = np.zeros((len(texts), 384), np.float32)
        order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
        for s in range(0, len(order), 32):
            b = order[s:s + 32]; coded = words.encode_batch([texts[i] for i in b])
            ids = np.array([c.ids for c in coded], np.int64)
            out[b] = session.run(None, {"input_ids": ids, "attention_mask": np.array([c.attention_mask for c in coded], np.int64),
                                        "token_type_ids": np.zeros_like(ids)})[0][:, 0]
        return out / np.maximum(np.linalg.norm(out, axis=1, keepdims=True), 1e-9)
    results[label + " · 8-bit"] = score(onnx_encode, label + " · 8-bit")
json.dump(results, open("bge-cip-v3/results.json", "w"), indent=1, default=str)
shutil.copy("synthetic.json", "bge-cip-v3/")
shutil.make_archive("bge-cip-v3", "zip", "bge-cip-v3")
print("READY: bge-cip-v3.zip is downloading to your Downloads folder.")
print("If the download does not start: left bar → folder icon → content → bge-cip-v3.zip → ⋮ → Download (within 15 minutes).")
files.download("bge-cip-v3.zip")

In [ ]:
# Stop paying once the download has had plenty of time: an idle GPU still uses units
import time
from google.colab import runtime
print("Releasing the GPU in 15 minutes. If bge-cip-v3.zip is not in Downloads yet: folder icon → content → bge-cip-v3.zip → ⋮ → Download")
time.sleep(900)
runtime.unassign()